# Power BI data preparation

This notebook summarises some of the tables to be used for visualisation in Power BI.

In [1]:
import duckdb as db
import pandas as pd
import geopandas as gpd

In [2]:
con = db.connect("../data/transport_project.duckdb")
con.execute("SELECT * FROM research_panel LIMIT 5").fetch_df()

,site_id,survey_date,year,month,total_count,time_0600,time_0700,time_0800,time_1600,time_1700,time_1800,longitude,latitude,cluster
0,2,2010-03-01,2010,March,140,7,12,22,29,32,38,151.208601,-33.878286,0
1,2,2010-10-01,2010,October,84,5,16,23,12,13,15,151.208601,-33.878286,0
2,2,2011-03-01,2011,March,106,19,22,12,20,22,11,151.208601,-33.878286,0
3,2,2011-10-01,2011,October,243,40,45,39,32,53,34,151.208601,-33.878286,0
4,2,2012-03-01,2012,March,299,12,23,28,48,112,76,151.208601,-33.878286,0


In [3]:
# main page to show summary of the survey
df_fact_bike_counts = \
con.execute("""
    SELECT
        site_id,
        survey_date, year, month,
        (CASE WHEN month = 'October' THEN 1 ELSE 0 END) AS is_october,
        total_count,
        (time_0600 + time_0700 + time_0800) AS morning_count,
        (time_1600 + time_1700 + time_1800) AS evening_count,
        cluster,
        latitude, longitude
    FROM research_panel
    ORDER BY site_id
""").fetch_df()

df_fact_bike_counts.head()

,site_id,survey_date,year,month,is_october,total_count,morning_count,evening_count,cluster,latitude,longitude
0,2,2010-03-01,2010,March,0,140,41,99,0,-33.878286,151.208601
1,2,2010-10-01,2010,October,1,84,44,40,0,-33.878286,151.208601
2,2,2011-03-01,2011,March,0,106,53,53,0,-33.878286,151.208601
3,2,2011-10-01,2011,October,1,243,124,119,0,-33.878286,151.208601
4,2,2012-03-01,2012,March,0,299,63,236,0,-33.878286,151.208601


In [4]:
con.execute("SELECT * FROM clustering_panel_balanced LIMIT 5").fetch_df()

,site_id,n_surveys,mean_total_count,std_total_count,mean_march_count,mean_october_count,mean_march_october_diff,mean_morning_count,mean_evening_count,morning_evening_ratio,longitude,latitude
0,2,25,463.32,225.640149,410.250000,512.307692,-82.083333,174.92,288.40,0.637270,151.208601,-33.878286
1,4,25,797.40,164.294451,829.583333,767.692308,67.333333,399.48,397.92,1.024202,151.209897,-33.873292
2,5,25,400.84,87.262382,404.333333,397.615385,9.333333,185.08,215.76,0.898551,151.209083,-33.879797
3,8,25,251.80,76.645287,239.916667,262.769231,-28.000000,104.56,147.24,0.715469,151.207321,-33.865170
4,10,25,669.56,301.195407,621.333333,714.076923,-60.666667,314.00,355.56,0.924173,151.205093,-33.872817


In [5]:
con.execute("SELECT * FROM bike_sites LIMIT 5").fetch_df()

,site_id,intersection,longitude,latitude
0,1,"Intersection of Broadway, Lee Street, Quay Str...",151.204168,-33.882846
1,2,Intersection of Castlereagh Street and Goulbur...,151.208601,-33.878286
2,3,Intersection of Grosvenor Street and Glouceste...,151.205783,-33.863364
3,4,Intersection of Elizabeth Street and Park Street,151.209897,-33.873292
4,5,"Intersection of Elizabeth Street, Wentworth Av...",151.209083,-33.879797


In [6]:
# bike sites information
df_dim_sites = \
con.execute("""
    WITH
        clusters AS (SELECT DISTINCT site_id, cluster FROM research_panel),
        temp AS(
            SELECT DISTINCT
                p.*, c.cluster
            FROM clustering_panel_balanced AS p
            INNER JOIN clusters AS c
                ON p.site_id = c.site_id
        )
    SELECT
        p.site_id, b.intersection AS site_name,
        p.longitude, p.latitude,
        p.cluster,
        p.mean_total_count, p.std_total_count,
        p.morning_evening_ratio,
        p.mean_march_october_diff
    FROM temp AS p
    INNER JOIN bike_sites as b
        ON p.site_id = b.site_id      
    ORDER BY site_id
""").fetchdf()

interpretation_map = {
    0: "Southwest peripheral sites",
    1: "Inner-city connector streets",
    2: "High-volume commuter corridors",
}

df_dim_sites["cluster_interpretation"] = df_dim_sites["cluster"].map(interpretation_map)
df_dim_sites.head()

,site_id,site_name,longitude,latitude,cluster,mean_total_count,std_total_count,morning_evening_ratio,mean_march_october_diff,cluster_interpretation
0,2,Intersection of Castlereagh Street and Goulbur...,151.208601,-33.878286,0,463.32,225.640149,0.637270,-82.083333,Southwest peripheral sites
1,4,Intersection of Elizabeth Street and Park Street,151.209897,-33.873292,1,797.40,164.294451,1.024202,67.333333,Inner-city connector streets
2,5,"Intersection of Elizabeth Street, Wentworth Av...",151.209083,-33.879797,1,400.84,87.262382,0.898551,9.333333,Inner-city connector streets
3,8,"Intersection of George Street, Margaret Street...",151.207321,-33.865170,1,251.80,76.645287,0.715469,-28.000000,Inner-city connector streets
4,10,Intersection of Kent Street and Druitt Street,151.205093,-33.872817,2,669.56,301.195407,0.924173,-60.666667,High-volume commuter corridors


In [7]:
df_cluster_trends = df_fact_bike_counts.groupby(["cluster", "survey_date"]).agg(
    mean_total_count = ('total_count', 'mean'),
    median_total_count = ('total_count', 'median'),
    site_count = ('site_id', 'count')
).reset_index()

df_cluster_trends.head()

,cluster,survey_date,mean_total_count,median_total_count,site_count
0,0,2010-03-01,195.333333,142.0,36
1,0,2010-10-01,241.972222,192.0,36
2,0,2011-03-01,253.916667,193.0,36
3,0,2011-10-01,294.861111,216.5,36
4,0,2012-03-01,325.083333,293.5,36


In [8]:
# df_fact_bike_counts.to_csv("./exports/fact_bike_counts.csv")
# df_dim_sites.to_csv("./exports/dim_sites.csv")
# df_cluster_trends.to_csv("./exports/cluster_trends.csv")